# Câu 39:
Viết thuật toán cho phương pháp Gauss / Gauss-Jordan tìm ma trận nghịch đảo của ma trận $A$. Áp dụng cho một ma trận vuông cỡ 8.

## Ma trận nghịch đảo bằng Gauss-Jordan

### Bài toán

Cho ma trận vuông khả nghịch $A$ cấp $n$. Tìm $A^{-1}$ bằng phép khử **Gauss-Jordan**.

### Cơ sở lý thuyết

Ghép ma trận mở rộng $[\,A \mid I\,]$ rồi dùng các **phép biến đổi hàng sơ cấp** đưa khối trái về ma trận đơn vị. Vì cùng dãy phép biến đổi tương đương với nhân trái bởi $A^{-1}$:

$$[\,A \mid I\,] \;\xrightarrow{\text{khử Gauss-Jordan}}\; [\,I \mid A^{-1}\,]$$

khối phải khi đó chính là $A^{-1}$.

---

### Thuật toán

**Đầu vào:** Ma trận $A$ cấp $n$

**Đầu ra:** $A^{-1}$ (hoặc báo suy biến)

**Bước 1.** Lập $M \leftarrow [\,A \mid I\,]$ (cấp $n \times 2n$).

**Bước 2 — Vòng lặp** $k = 1, \ldots, n$:

&emsp;**2.1.** *(Pivot từng phần)* Chọn hàng $p \geq k$ có $|m_{pk}|$ lớn nhất; nếu $\approx 0$ ⟹ **suy biến, dừng**. Hoán vị hàng $k \leftrightarrow p$.

&emsp;**2.2.** Chuẩn hoá hàng trụ: $M[k, :] \leftarrow M[k, :] / m_{kk}$.

&emsp;**2.3.** Khử mọi hàng $i \neq k$: $M[i, :] \leftarrow M[i, :] - m_{ik} \, M[k, :]$.

**Bước 3.** Trả về $A^{-1} = M[:, \, n{:}\,2n]$ (khối phải).

---

### Lưu ý

- **Gauss vs Gauss-Jordan:** Gauss thường chỉ khử xuống dưới (tam giác trên) rồi thế ngược; **Gauss-Jordan** khử cả trên lẫn dưới để ra thẳng $I$ — tốn hơn chút ($\sim n^3$ so với $\tfrac{2}{3}n^3$ của khử Gauss) nhưng cho nghịch đảo trực tiếp, không cần thế ngược.
- **Pivot từng phần (Bước 2.1) là bắt buộc** cho ổn định số học: tránh chia cho trụ nhỏ khuếch đại sai số làm tròn.
- **Kiểm tra:** $\|A A^{-1} - I\|_\infty \approx 0$ và đối chiếu với `numpy.linalg.inv`.
- **Khi nào KHÔNG nên tính nghịch đảo:** để **giải** $Ax=b$, dùng phân tách LU rẻ và ổn định hơn tính $A^{-1}$ rồi nhân. Nghịch đảo chỉ cần khi thực sự dùng lại nhiều lần hoặc phân tích lý thuyết.

In [ ]:
import numpy as np


# ── Định dạng số khoa học kiểu 1.23×10⁻⁵ ─────────────────────────
def _sci(x, sig=4):
    if abs(x) < 1e-14:
        return "0"
    s = f"{x:.{sig}e}"
    m, e = s.split("e")
    exp = int(e)
    sup = str(abs(exp)).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    sign = "⁻" if exp < 0 else ""
    return f"{m}×10{sign}{sup}"


def _print_mat(M, label, w=8, p=4):
    print(f"\n{label}:")
    for row in M:
        print("  " + "  ".join(f"{v:{w}.{p}f}" for v in row))


# ── Nghịch đảo bằng Gauss-Jordan (có pivot từng phần) ────────────
def inv_gauss_jordan(A, log=True):
    """
    Trả về (A⁻¹, info). Dùng ma trận mở rộng [A | I] → [I | A⁻¹].
    """
    A = np.array(A, dtype=float)
    n = len(A)
    M = np.hstack([A.copy(), np.eye(n)])       # [A | I]

    for k in range(n):
        # pivot từng phần: hàng có |phần tử cột k| lớn nhất
        p = k + int(np.argmax(np.abs(M[k:, k])))
        if abs(M[p, k]) < 1e-14:
            return None, "Ma trận suy biến"
        if p != k:
            M[[k, p]] = M[[p, k]]

        M[k] = M[k] / M[k, k]                   # chuẩn hoá hàng trụ
        for i in range(n):                      # khử mọi hàng khác
            if i != k:
                M[i] = M[i] - M[i, k] * M[k]

        if log and k in (0, n // 2, n - 1):
            print(f"  Bước k={k+1:>2}: trụ sau chuẩn hoá = {_sci(M[k, k])}")

    return M[:, n:], "ok"


# ====== Áp dụng với ma trận vuông cỡ 8 ======
if __name__ == "__main__":
    # Ma trận cấp 8 (chéo trội — dùng lại cho Câu 42a)
    A = np.array([
        [  23.,    0.,   -3.,    1.,    3.,   -5.,   -3.,   -3.],
        [   4.,   34.,   -1.,    4.,    6.,   -5.,    5.,   -4.],
        [   2.,    6.,   22.,    2.,   -2.,    3.,    0.,    2.],
        [  -1.,   -4.,    2.,   25.,   -3.,   -4.,    6.,    0.],
        [  -4.,    1.,   -2.,    6.,   28.,    4.,   -1.,   -5.],
        [  -4.,    5.,    4.,   -3.,    1.,   25.,    2.,   -1.],
        [   3.,    4.,   -4.,    4.,    4.,    4.,   28.,    0.],
        [  -2.,   -4.,    0.,   -2.,   -3.,    1.,   -1.,   18.],
    ])
    n = len(A)

    print(f"Ma trận A cấp {n},  det(A) = {np.linalg.det(A):.4g}")
    Ainv, info = inv_gauss_jordan(A)
    print(f"\nTrạng thái: {info}")

    _print_mat(Ainv, "Ma trận nghịch đảo A⁻¹")

    # ── Kiểm tra ──
    err_I  = np.max(np.abs(A @ Ainv - np.eye(n)))
    err_np = np.max(np.abs(Ainv - np.linalg.inv(A)))
    print("\n=== Kiểm tra ===")
    print(f"  ‖A·A⁻¹ - I‖∞        = {_sci(err_I)}")
    print(f"  Sai số vs numpy.inv = {_sci(err_np)}"
          f"  →  {'ĐÚNG ✓' if err_np < 1e-8 else 'SAI ✗'}")


# Câu 40:
Viết thuật toán cho phương pháp phân tách Choleski tìm ma trận nghịch đảo của ma trận vuông $A$. Áp dụng cho một ma trận vuông cấp 5.

## Ma trận nghịch đảo bằng phân tách Choleski

### Bài toán

Cho ma trận **đối xứng xác định dương** $A$ cấp $n$. Tìm $A^{-1}$ qua phân tách Choleski.

### Cơ sở lý thuyết

**Phân tách Choleski:** ma trận đối xứng xác định dương phân tích duy nhất thành

$$A = L L^T$$

với $L$ tam giác dưới, đường chéo dương. Công thức tính từng phần tử:

$$L_{jj} = \sqrt{A_{jj} - \sum_{k<j} L_{jk}^2}, \qquad L_{ij} = \frac{1}{L_{jj}}\Big( A_{ij} - \sum_{k<j} L_{ik} L_{jk} \Big), \quad i > j$$

**Tìm nghịch đảo:** giải $n$ hệ $A x_j = e_j$ ($e_j$ là cột $j$ của $I$); nghiệm $x_j$ là cột $j$ của $A^{-1}$. Mỗi hệ giải qua hai bước tam giác:

$$L y = e_j \;\;(\text{thế xuôi}), \qquad L^T x_j = y \;\;(\text{thế ngược})$$

---

### Thuật toán

**Đầu vào:** Ma trận đối xứng xác định dương $A$ cấp $n$

**Đầu ra:** $A^{-1}$

**Bước 1 — Phân tách:** tính $L$ sao cho $A = LL^T$. Nếu gặp $A_{jj} - \sum L_{jk}^2 \le 0$ ⟹ **không xác định dương, dừng**.

**Bước 2 — Nghịch đảo:** với mỗi $j = 1, \ldots, n$:

&emsp;**2.1.** $y \leftarrow$ giải $L y = e_j$ (thế xuôi).

&emsp;**2.2.** cột $j$ của $A^{-1} \leftarrow$ giải $L^T x = y$ (thế ngược).

**Bước 3 — Kiểm tra:** $A A^{-1} \approx I$.

---

### Lưu ý

- **Điều kiện áp dụng:** Choleski **chỉ dùng** cho ma trận đối xứng xác định dương. Nếu $A$ không đối xứng hoặc không xác định dương, phải dùng LU (Gauss) — Bước 1 sẽ báo lỗi khi gặp căn của số $\le 0$.
- **Rẻ gấp đôi LU:** Choleski chỉ tốn $\tfrac{1}{3}n^3$ phép tính (so với $\tfrac{2}{3}n^3$ của LU) vì khai thác tính đối xứng — chỉ tính nửa dưới.
- **Ổn định không cần pivot:** Với ma trận xác định dương, Choleski **ổn định số học** mà không cần chọn trụ — một ưu điểm lớn.
- **Tận dụng lại $L$:** Phân tách một lần, giải $n$ hệ cho $n$ cột nghịch đảo — không phân tách lại mỗi cột.

In [ ]:
import numpy as np


# ── Định dạng số khoa học kiểu 1.23×10⁻⁵ ─────────────────────────
def _sci(x, sig=4):
    if abs(x) < 1e-14:
        return "0"
    s = f"{x:.{sig}e}"
    m, e = s.split("e")
    exp = int(e)
    sup = str(abs(exp)).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    sign = "⁻" if exp < 0 else ""
    return f"{m}×10{sign}{sup}"


def _print_mat(M, label, w=8, p=4):
    print(f"\n{label}:")
    for row in M:
        print("  " + "  ".join(f"{v:{w}.{p}f}" for v in row))


# ── Phân tách Choleski A = L Lᵀ ──────────────────────────────────
def cholesky(A):
    A = np.array(A, dtype=float)
    n = len(A)
    L = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1):
            s = A[i, j] - L[i, :j] @ L[j, :j]
            if i == j:
                if s <= 0:
                    return None, "A không xác định dương"
                L[i, j] = np.sqrt(s)
            else:
                L[i, j] = s / L[j, j]
    return L, "ok"


def forward_sub(L, b):        # L y = b (tam giác dưới)
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]
    return y


def back_sub_T(L, y):         # Lᵀ x = y (tam giác trên)
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - L[i+1:, i] @ x[i+1:]) / L[i, i]
    return x


# ── Nghịch đảo bằng Choleski ─────────────────────────────────────
def inv_cholesky(A):
    L, info = cholesky(A)
    if info != "ok":
        return None, None, info
    n = len(A)
    Inv = np.zeros((n, n))
    for j in range(n):                     # giải A x_j = e_j
        e = np.zeros(n); e[j] = 1.0
        y = forward_sub(L, e)              # L y = e_j
        Inv[:, j] = back_sub_T(L, y)       # Lᵀ x = y
    return Inv, L, "ok"


# ====== Áp dụng với ma trận đối xứng xác định dương cấp 5 ======
if __name__ == "__main__":
    A = np.array([
        [ 19.,   3.,   3.,   1., -18.],
        [  3.,  33., -24.,  -7.,   5.],
        [  3., -24.,  33.,   1.,  -9.],
        [  1.,  -7.,   1.,  16., -10.],
        [-18.,   5.,  -9., -10.,  37.],
    ])
    n = len(A)

    print("A đối xứng?  ", np.allclose(A, A.T))
    Ainv, L, info = inv_cholesky(A)
    print("Trạng thái:  ", info)

    _print_mat(L, "Nhân tử Choleski L (A = L Lᵀ)")
    print(f"\n  Kiểm tra phân tách: ‖L Lᵀ - A‖ = {_sci(np.max(np.abs(L @ L.T - A)))}")

    _print_mat(Ainv, "Ma trận nghịch đảo A⁻¹")

    # ── Kiểm tra ──
    err_I  = np.max(np.abs(A @ Ainv - np.eye(n)))
    err_np = np.max(np.abs(Ainv - np.linalg.inv(A)))
    print("\n=== Kiểm tra ===")
    print(f"  ‖A·A⁻¹ - I‖∞        = {_sci(err_I)}")
    print(f"  Sai số vs numpy.inv = {_sci(err_np)}"
          f"  →  {'ĐÚNG ✓' if err_np < 1e-8 else 'SAI ✗'}")


# Câu 41:
Viết thuật toán cho phương pháp viền quanh tìm ma trận nghịch đảo. Áp dụng cho một ma trận cấp 7.

## Ma trận nghịch đảo bằng phương pháp viền quanh

### Bài toán

Cho ma trận vuông khả nghịch $A$ cấp $n$. Tìm $A^{-1}$ bằng phương pháp **viền quanh** (bordering): xây nghịch đảo **tăng dần theo cấp**, từ khối $1\times1$ đến cấp $n$.

### Cơ sở lý thuyết

Giả sử đã biết nghịch đảo của khối con $A_k$ (góc trên-trái cấp $k$). Viền thêm một hàng và một cột để có $A_{k+1}$:

$$A_{k+1} = \begin{bmatrix} A_k & u \\ v^T & \alpha \end{bmatrix}$$

với $u = A[1{:}k,\, k+1]$ (cột viền), $v^T = A[k+1,\, 1{:}k]$ (hàng viền), $\alpha = A_{k+1,k+1}$.

Đặt **bù Schur** $\theta = \alpha - v^T A_k^{-1} u$. Khi $\theta \neq 0$:

$$A_{k+1}^{-1} = \begin{bmatrix}
A_k^{-1} + \dfrac{1}{\theta}(A_k^{-1}u)(v^T A_k^{-1}) & -\dfrac{1}{\theta}A_k^{-1}u \\[2mm]
-\dfrac{1}{\theta}v^T A_k^{-1} & \dfrac{1}{\theta}
\end{bmatrix}$$

Công thức này **không cần nghịch đảo lại** — chỉ dùng $A_k^{-1}$ đã có và vài phép nhân ma trận–vector.

---

### Thuật toán

**Đầu vào:** Ma trận $A$ cấp $n$

**Đầu ra:** $A^{-1}$

**Bước 1.** Khởi tạo khối $1\times1$: $A_1^{-1} = [\,1/a_{11}\,]$ (nếu $a_{11}=0$ ⟹ cần hoán vị / dừng).

**Bước 2 — Vòng lặp** $k = 1, \ldots, n-1$:

&emsp;**2.1.** Lấy viền $u, v^T, \alpha$ của $A_{k+1}$.

&emsp;**2.2.** $p \leftarrow A_k^{-1} u$, &nbsp; $q^T \leftarrow v^T A_k^{-1}$, &nbsp; $\theta \leftarrow \alpha - v^T p$.

&emsp;**2.3.** Nếu $\theta \approx 0$ ⟹ **khối con suy biến, dừng**.

&emsp;**2.4.** Ghép $A_{k+1}^{-1}$ theo công thức trên.

**Bước 3.** Trả về $A_n^{-1} = A^{-1}$.

---

### Lưu ý

- **Ưu điểm cập nhật:** Mỗi bước chỉ tốn $O(k^2)$ (nhân ma trận–vector), tổng cộng $O(n^3)$ — cùng bậc với Gauss, nhưng **thêm được một hàng/cột** mà không tính lại từ đầu. Rất hữu ích khi ma trận **lớn dần** theo thời gian (thêm biến/quan sát).
- **Điều kiện $\theta \neq 0$:** Bù Schur $\theta = 0$ nghĩa là khối con $A_{k+1}$ suy biến (dù $A$ toàn phần có thể không). Khi đó cần hoán vị thứ tự hàng/cột để mọi khối con đầu đều khả nghịch.
- **Quan hệ với bù Schur:** $\det A_{k+1} = \theta \cdot \det A_k$, nên tích các $\theta$ cho $\det A$ — có thể tận dụng để tính định thức kèm theo.
- **Kiểm tra:** $\|A A^{-1} - I\|_\infty \approx 0$, đối chiếu `numpy.linalg.inv`.

In [ ]:
import numpy as np


# ── Định dạng số khoa học kiểu 1.23×10⁻⁵ ─────────────────────────
def _sci(x, sig=4):
    if abs(x) < 1e-14:
        return "0"
    s = f"{x:.{sig}e}"
    m, e = s.split("e")
    exp = int(e)
    sup = str(abs(exp)).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    sign = "⁻" if exp < 0 else ""
    return f"{m}×10{sign}{sup}"


def _print_mat(M, label, w=8, p=4):
    print(f"\n{label}:")
    for row in M:
        print("  " + "  ".join(f"{v:{w}.{p}f}" for v in row))


# ── Nghịch đảo bằng viền quanh (bordering) ───────────────────────
def inv_bordering(A, log=True):
    """
    Xây A⁻¹ tăng dần theo cấp k. Trả về (A⁻¹, info).
    """
    A = np.array(A, dtype=float)
    n = len(A)
    if abs(A[0, 0]) < 1e-14:
        return None, "a₁₁ = 0, cần hoán vị"

    Inv = np.array([[1.0 / A[0, 0]]])          # khối 1×1

    for k in range(1, n):
        Ak_inv = Inv
        u = A[:k, k].reshape(-1, 1)            # cột viền
        v = A[k, :k].reshape(1, -1)            # hàng viền
        alpha = A[k, k]

        p = Ak_inv @ u                         # A_k⁻¹ u
        q = v @ Ak_inv                         # vᵀ A_k⁻¹
        theta = alpha - (v @ p)[0, 0]          # bù Schur
        if abs(theta) < 1e-14:
            return None, f"Bù Schur = 0 tại k={k+1}"

        top_left  = Ak_inv + (p @ q) / theta
        top_right = -p / theta
        bot_left  = -q / theta
        bot_right = np.array([[1.0 / theta]])
        Inv = np.block([[top_left, top_right],
                        [bot_left, bot_right]])

        if log and k in (1, n // 2, n - 1):
            print(f"  Cấp {k+1}: bù Schur θ = {_sci(theta)}")

    return Inv, "ok"


# ====== Áp dụng với ma trận cấp 7 ======
if __name__ == "__main__":
    A = np.array([
        [ 11.,   5.,   4.,   3.,   2.,  -4.,  -3.],
        [  4.,   7.,   4.,  -4.,  -2.,   5.,   0.],
        [  4.,   1.,   4.,  -2.,   0.,   1.,   5.],
        [  3.,  -2.,   4.,   8.,  -2.,  -5.,  -1.],
        [ -3.,   1.,   2.,   5.,   9.,   5.,   5.],
        [ -5.,   3.,   3.,   2.,  -4.,   8.,  -2.],
        [  5.,   1.,   2.,   3.,   2.,  -4.,   5.],
    ])
    n = len(A)

    print(f"Ma trận A cấp {n},  det(A) = {np.linalg.det(A):.4g}")
    Ainv, info = inv_bordering(A)
    print(f"\nTrạng thái: {info}")

    _print_mat(Ainv, "Ma trận nghịch đảo A⁻¹")

    # ── Kiểm tra ──
    err_I  = np.max(np.abs(A @ Ainv - np.eye(n)))
    err_np = np.max(np.abs(Ainv - np.linalg.inv(A)))
    print("\n=== Kiểm tra ===")
    print(f"  ‖A·A⁻¹ - I‖∞        = {_sci(err_I)}")
    print(f"  Sai số vs numpy.inv = {_sci(err_np)}"
          f"  →  {'ĐÚNG ✓' if err_np < 1e-8 else 'SAI ✗'}")


# Câu 42a:
Viết thuật toán tìm ma trận nghịch đảo của ma trận $A$ bằng phương pháp lặp Jacobi / lặp Gauss-Seidel với sai số tuyệt đối / sai số tương đối cho trước. Áp dụng với ma trận cấp 8 ở câu 39 và so sánh.

## Ma trận nghịch đảo bằng lặp Jacobi / Gauss-Seidel

### Bài toán

Cho ma trận $A$ cấp $n$ (chéo trội để đảm bảo hội tụ). Tìm $A^{-1}$ bằng phương pháp **lặp** — giải $n$ hệ $A x_j = e_j$ bằng Jacobi hoặc Gauss-Seidel, mỗi nghiệm $x_j$ là một cột của $A^{-1}$.

### Cơ sở lý thuyết

Tìm nghịch đảo tương đương giải phương trình ma trận $A X = I$. Tách theo cột: cột $j$ của $X$ là nghiệm của $A x_j = e_j$. Áp dụng công thức lặp cho từng hệ:

- **Jacobi:** $\;x_i^{(k+1)} = \dfrac{1}{a_{ii}}\Big(e_{ij} - \sum_{l \neq i} a_{il} x_l^{(k)}\Big)$ — dùng toàn bộ $x^{(k)}$ cũ.
- **Gauss-Seidel:** dùng ngay giá trị vừa cập nhật trong cùng vòng ⟹ hội tụ nhanh hơn.

Điều kiện hội tụ (đủ): $A$ **chéo trội nghiêm ngặt** $|a_{ii}| > \sum_{j\neq i}|a_{ij}|$.

---

### Thuật toán

**Đầu vào:** Ma trận $A$ cấp $n$, sai số $\varepsilon$, phương pháp (Jacobi / GS)

**Đầu ra:** $A^{-1}$ và số vòng lặp mỗi cột

**Bước 1.** Kiểm tra chéo trội (điều kiện đủ hội tụ).

**Bước 2 — Vòng lặp** $j = 1, \ldots, n$:

&emsp;Giải $A x_j = e_j$ bằng phương pháp đã chọn đến khi $\|x^{(k+1)} - x^{(k)}\|_\infty < \varepsilon$; gán cột $j$ của $A^{-1} \leftarrow x_j$.

**Bước 3 — So sánh & kiểm tra.** Đối chiếu tổng số vòng lặp Jacobi vs Gauss-Seidel; kiểm tra $\|A A^{-1} - I\| \approx 0$.

---

### So sánh & Lưu ý

- **Gauss-Seidel nhanh hơn Jacobi** (thường ~gấp đôi tốc độ) vì dùng ngay giá trị mới cập nhật — thấy rõ ở tổng số vòng lặp trong kết quả bên dưới.
- **Chỉ hợp ma trận chéo trội / xác định dương:** với ma trận thường không hội tụ. Ma trận cấp 8 ở Câu 39 được chọn chéo trội chính vì vậy.
- **So với phương pháp trực tiếp (Câu 39, 40, 41):** lặp cho **nghiệm xấp xỉ** (sai số $\sim\varepsilon$), trong khi Gauss/Choleski/viền quanh cho nghiệm **chính xác** (đến sai số máy). Lặp chỉ lợi khi ma trận **thưa** và lớn.
- **Song song hoá:** Jacobi tính các thành phần độc lập ⟹ dễ song song; Gauss-Seidel tuần tự.

In [ ]:
import numpy as np


# ── Định dạng số khoa học kiểu 1.23×10⁻⁵ ─────────────────────────
def _sci(x, sig=4):
    if abs(x) < 1e-14:
        return "0"
    s = f"{x:.{sig}e}"
    m, e = s.split("e")
    exp = int(e)
    sup = str(abs(exp)).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    sign = "⁻" if exp < 0 else ""
    return f"{m}×10{sign}{sup}"


# ── Giải Ax=b bằng Jacobi ────────────────────────────────────────
def solve_jacobi(A, b, eps=1e-8, max_iter=5000):
    A, b = np.array(A, float), np.array(b, float)
    d = np.diag(A)
    x = np.zeros(len(b))
    for k in range(1, max_iter + 1):
        x_new = (b - (A @ x - d * x)) / d       # dùng x cũ toàn bộ
        if np.max(np.abs(x_new - x)) < eps:
            return x_new, k
        x = x_new
    return x, max_iter


# ── Giải Ax=b bằng Gauss-Seidel ──────────────────────────────────
def solve_gauss_seidel(A, b, eps=1e-8, max_iter=5000):
    A, b = np.array(A, float), np.array(b, float)
    n = len(b)
    x = np.zeros(n)
    for k in range(1, max_iter + 1):
        x_old = x.copy()
        for i in range(n):
            s1 = A[i, :i] @ x[:i]                # x mới
            s2 = A[i, i+1:] @ x_old[i+1:]        # x cũ
            x[i] = (b[i] - s1 - s2) / A[i, i]
        if np.max(np.abs(x - x_old)) < eps:
            return x, k
    return x, max_iter


# ── Nghịch đảo bằng lặp: giải A x_j = e_j cho mỗi cột ────────────
def inv_iterative(A, method, eps=1e-8):
    n = len(A)
    Inv = np.zeros((n, n))
    iters = []
    for j in range(n):
        e = np.zeros(n); e[j] = 1.0
        xj, k = method(A, e, eps)
        Inv[:, j] = xj
        iters.append(k)
    return Inv, iters


# ====== Áp dụng với ma trận cấp 8 (Câu 39) ======
if __name__ == "__main__":
    A = np.array([
        [  23.,    0.,   -3.,    1.,    3.,   -5.,   -3.,   -3.],
        [   4.,   34.,   -1.,    4.,    6.,   -5.,    5.,   -4.],
        [   2.,    6.,   22.,    2.,   -2.,    3.,    0.,    2.],
        [  -1.,   -4.,    2.,   25.,   -3.,   -4.,    6.,    0.],
        [  -4.,    1.,   -2.,    6.,   28.,    4.,   -1.,   -5.],
        [  -4.,    5.,    4.,   -3.,    1.,   25.,    2.,   -1.],
        [   3.,    4.,   -4.,    4.,    4.,    4.,   28.,    0.],
        [  -2.,   -4.,    0.,   -2.,   -3.,    1.,   -1.,   18.],
    ])
    n = len(A)

    # kiểm tra chéo trội
    off = np.sum(np.abs(A - np.diag(np.diag(A))), axis=1)
    print(f"A chéo trội nghiêm ngặt: {np.all(np.abs(np.diag(A)) > off)}\n")

    eps = 1e-8
    print(f"{'Phương pháp':<14}  {'tổng vòng':>9}  {'tb/cột':>7}  {'‖A·A⁻¹-I‖∞':>13}")
    print("─" * 52)
    results = {}
    for name, meth in [("Jacobi", solve_jacobi), ("Gauss-Seidel", solve_gauss_seidel)]:
        Inv, iters = inv_iterative(A, meth, eps=eps)
        err = np.max(np.abs(A @ Inv - np.eye(n)))
        results[name] = (Inv, sum(iters))
        print(f"{name:<14}  {sum(iters):>9}  {np.mean(iters):>7.1f}  {_sci(err):>13}")
    print("─" * 52)

    # ── So sánh & kiểm tra với numpy ──
    jt, gt = results["Jacobi"][1], results["Gauss-Seidel"][1]
    print(f"\nGauss-Seidel / Jacobi (số vòng) = {gt}/{jt} = {gt/jt:.2f}"
          f"  →  GS nhanh hơn ~{jt/gt:.1f} lần")
    err_np = np.max(np.abs(results["Gauss-Seidel"][0] - np.linalg.inv(A)))
    print(f"Sai số A⁻¹ (Gauss-Seidel) vs numpy = {_sci(err_np)}"
          f"  →  {'ĐÚNG ✓' if err_np < 1e-6 else 'SAI ✗'}")


# Câu 42b:
Viết thuật toán tìm ma trận nghịch đảo của $A$ bằng phương pháp lặp tựa Newton với sai số tuyệt đối / sai số tương đối cho trước. Áp dụng với ma trận cấp 7 ở câu 41 và so sánh.

## Ma trận nghịch đảo bằng lặp tựa Newton (Newton–Schulz)

### Bài toán

Cho ma trận vuông khả nghịch $A$ cấp $n$. Tìm $A^{-1}$ bằng phương pháp **lặp tựa Newton** — áp dụng lặp Newton cho phương trình ma trận $F(X) = X^{-1} - A = 0$.

### Cơ sở lý thuyết

Áp dụng phương pháp Newton cho $F(X) = X^{-1} - A$ dẫn tới công thức lặp **Newton–Schulz** (không cần nghịch đảo ở mỗi bước):

$$X_{k+1} = X_k \big( 2I - A X_k \big)$$

**Hội tụ bậc hai:** đặt $R_k = I - A X_k$, ta có $R_{k+1} = R_k^2$, nên $\|R_{k+1}\| \le \|R_k\|^2$. Sai số **bình phương** mỗi vòng — một khi đã vào vùng hội tụ ($\|R_k\| < 1$) thì rất nhanh.

**Khởi tạo đảm bảo hội tụ:**

$$X_0 = \frac{A^T}{\|A\|_1 \, \|A\|_\infty}$$

đảm bảo $\|I - A X_0\|_2 < 1$, tức residual ban đầu đã nằm trong vùng hội tụ.

---

### Thuật toán

**Đầu vào:** Ma trận $A$ cấp $n$, sai số $\varepsilon$, số vòng tối đa $N$

**Đầu ra:** $X \approx A^{-1}$

**Bước 1.** Khởi tạo $X \leftarrow A^T / (\|A\|_1 \|A\|_\infty)$.

**Bước 2 — Vòng lặp** $k = 1, \ldots, N$:

&emsp;**2.1.** $R \leftarrow I - A X$; sai số $r \leftarrow \|R\|_\infty$.

&emsp;**2.2.** $X \leftarrow X(2I - A X)$.

&emsp;**2.3.** Nếu $r < \varepsilon$: **dừng**.

**Bước 3 — So sánh & kiểm tra** với kết quả trực tiếp (Câu 41) và `numpy.inv`.

---

### So sánh & Lưu ý

- **Chỉ dùng phép nhân ma trận:** mỗi vòng là 2 phép nhân $n\times n$ ($O(n^3)$ mỗi vòng). Không có phép chia/khử ⟹ **rất hợp cho GPU / tính song song** và cho ma trận thưa.
- **Hai giai đoạn hội tụ:** ban đầu residual giảm chậm (giai đoạn tuyến tính khi $\|R\|$ còn gần 1), sau đó **bình phương** rất nhanh khi $\|R\| < 1$ — thấy rõ trong log bên dưới.
- **So với phương pháp trực tiếp (viền quanh, Câu 41):** trực tiếp cho nghiệm sau đúng $n$ bước xác định; tựa Newton cần nhiều vòng hơn nhưng **mỗi vòng đơn giản** và **tự sửa sai số** (self-correcting — sai số làm tròn không tích luỹ vì mỗi bước là một bước Newton mới).
- **Nhạy khởi tạo:** nếu $X_0$ không thoả $\|I - AX_0\| < 1$ (với một chuẩn ma trận nào đó), phương pháp **phân kỳ**. Công thức khởi tạo ở trên là điều kiện đủ an toàn.

In [ ]:
import numpy as np


# ── Định dạng số khoa học kiểu 1.23×10⁻⁵ ─────────────────────────
def _sci(x, sig=4):
    if abs(x) < 1e-14:
        return "0"
    s = f"{x:.{sig}e}"
    m, e = s.split("e")
    exp = int(e)
    sup = str(abs(exp)).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    sign = "⁻" if exp < 0 else ""
    return f"{m}×10{sign}{sup}"


# ── Nghịch đảo bằng lặp tựa Newton (Newton–Schulz) ───────────────
def inv_newton(A, eps=1e-12, max_iter=100, log=True):
    """
    X_{k+1} = X_k (2I - A X_k),  hội tụ bậc hai về A⁻¹.
    Trả về (X, số vòng).
    """
    A = np.array(A, dtype=float)
    n = len(A)
    I = np.eye(n)
    # khởi tạo đảm bảo ‖I - A X₀‖₂ < 1
    X = A.T / (np.linalg.norm(A, 1) * np.linalg.norm(A, np.inf))

    if log:
        print(f"{'k':>3}  {'‖I - A·X‖∞':>14}")
    for k in range(1, max_iter + 1):
        R = I - A @ X                          # residual
        r = np.max(np.abs(R))
        X = X @ (2 * I - A @ X)                # bước Newton–Schulz
        if log and (k <= 8 or k % 4 == 0):
            print(f"{k:>3}  {_sci(r):>14}")
        if r < eps:
            return X, k
    return X, max_iter


# ── (để so sánh) nghịch đảo trực tiếp bằng viền quanh — Câu 41 ────
def inv_bordering(A):
    A = np.array(A, dtype=float); n = len(A)
    Inv = np.array([[1.0 / A[0, 0]]])
    for k in range(1, n):
        u = A[:k, k].reshape(-1, 1); v = A[k, :k].reshape(1, -1)
        p = Inv @ u; q = v @ Inv
        theta = A[k, k] - (v @ p)[0, 0]
        Inv = np.block([[Inv + (p @ q) / theta, -p / theta],
                        [-q / theta, np.array([[1.0 / theta]])]])
    return Inv


# ====== Áp dụng với ma trận cấp 7 (Câu 41) ======
if __name__ == "__main__":
    A = np.array([
        [ 11.,   5.,   4.,   3.,   2.,  -4.,  -3.],
        [  4.,   7.,   4.,  -4.,  -2.,   5.,   0.],
        [  4.,   1.,   4.,  -2.,   0.,   1.,   5.],
        [  3.,  -2.,   4.,   8.,  -2.,  -5.,  -1.],
        [ -3.,   1.,   2.,   5.,   9.,   5.,   5.],
        [ -5.,   3.,   3.,   2.,  -4.,   8.,  -2.],
        [  5.,   1.,   2.,   3.,   2.,  -4.,   5.],
    ])
    n = len(A)

    print("Lặp tựa Newton (Newton–Schulz):")
    X, k = inv_newton(A, eps=1e-12)
    print(f"\nSố vòng lặp: {k}")

    # ── So sánh & kiểm tra ──
    err_I  = np.max(np.abs(A @ X - np.eye(n)))
    err_np = np.max(np.abs(X - np.linalg.inv(A)))
    X_direct = inv_bordering(A)                 # phương pháp trực tiếp Câu 41
    err_dir = np.max(np.abs(X - X_direct))

    print("\n=== So sánh & kiểm tra ===")
    print(f"  ‖A·X - I‖∞                    = {_sci(err_I)}")
    print(f"  Sai số vs numpy.inv           = {_sci(err_np)}"
          f"  →  {'ĐÚNG ✓' if err_np < 1e-8 else 'SAI ✗'}")
    print(f"  Sai số vs viền quanh (Câu 41) = {_sci(err_dir)}   (hai phương pháp khớp nhau)")
    print(f"\n  Nhận xét: tựa Newton cần {k} vòng (mỗi vòng 2 phép nhân n×n),")
    print(f"  còn viền quanh chỉ {n} bước trực tiếp — nhưng tựa Newton tự sửa sai số")
    print(f"  và dễ song song hoá. Cả hai cho cùng kết quả tới sai số máy.")


# Câu 42c:
Chọn một phương pháp và trình bày thuật toán tương ứng với phương pháp đã chọn để tìm ma trận nghịch đảo của một ma trận vuông $A$ với sai số không vượt quá $\varepsilon$ cho trước.

Chạy chương trình với ma trận $A$ dưới đây, ghi lại ít nhất một giá trị trung gian và kết quả cuối cùng tìm được, đánh giá sai số, kiểm tra, nhận xét.

$$
A = \begin{bmatrix}
11+a & 22 & -13 & 24 & 15 & -26 & 17 & 28 \\
22 & 233+a & 24 & 35 & 26 & 37 & 28 & -39 \\
33 & -24 & 35+a & -26 & 37 & 28 & -39 & 20 \\
14 & 45 & 26 & 47+a & 38 & 49 & 40 & -41 \\
-55 & 16 & 57 & 28 & 59+a & 30 & -51 & 42 \\
46 & 27 & -48 & 39 & 40 & 61+a & 42 & 73 \\
27 & -58 & 29 & 70 & -21 & 42 & 23+a & 34 \\
38 & 59 & 60 & -71 & 82 & -93 & 24 & 15+a
\end{bmatrix}, \qquad a = 200
$$

## Chọn phương pháp & chạy trên ma trận cấp 8 ($a=200$)

### Phân tích ma trận để chọn phương pháp

Ma trận $A = C + 200 I$ (cấp 8) có các tính chất:

| Tính chất | Kết quả | Hệ quả |
|---|---|---|
| Đối xứng? | **Không** | Loại **Choleski** (Câu 40) |
| Chéo trội nghiêm ngặt? | **Không** (hàng 4–8 vi phạm) | Jacobi/GS **không đảm bảo** hội tụ (chỉ là điều kiện đủ) |
| Số điều kiện $\kappa_2(A)$ | $\approx 4.67$ (rất tốt) | Lặp tựa Newton hội tụ **nhanh** |

**Phương pháp chọn: lặp tựa Newton (Newton–Schulz)** — vì:

- Hội tụ **đảm bảo** với khởi tạo $X_0 = A^T/(\|A\|_1\|A\|_\infty)$, **không phụ thuộc** tính chéo trội hay đối xứng.
- Hội tụ **bậc hai**; với $\kappa$ nhỏ như đây chỉ cần ~10 vòng.
- Cho phép dừng theo **sai số $\varepsilon$** đúng như đề yêu cầu, và **tự sửa sai số** làm tròn.

Công thức lặp (xem Câu 42b):

$$X_{k+1} = X_k(2I - A X_k), \qquad R_{k+1} = R_k^2$$

---

### Thuật toán (tựa Newton, dừng theo $\varepsilon$)

**Bước 1.** $X \leftarrow A^T / (\|A\|_1 \|A\|_\infty)$.

**Bước 2.** Lặp: $R \leftarrow I - AX$; nếu $\|R\|_\infty < \varepsilon$ dừng; ngược lại $X \leftarrow X(2I - AX)$.

**Bước 3.** Kiểm tra $\|AX - I\| \approx 0$, đối chiếu `numpy.inv`.

---

### Nhận xét

- **Ma trận không chéo trội nhưng lặp vẫn hội tụ:** kiểm tra trực tiếp cho $\rho(T_{\text{Jacobi}}) \approx 0.52 < 1$ và $\rho(T_{\text{GS}}) \approx 0.33 < 1$. Đây là minh hoạ rõ: **chéo trội chỉ là điều kiện đủ**, không cần thiết — cái quyết định hội tụ là bán kính phổ của ma trận lặp.

- **Vì sao vẫn ưu tiên tựa Newton:** dù Jacobi/GS tình cờ hội tụ ở đây, ta **không biết trước** điều đó nếu chỉ nhìn ma trận (phải tính $\rho(T)$). Tựa Newton có **bảo đảm lý thuyết** với khởi tạo an toàn, nên là lựa chọn chắc chắn hơn.

- **Kiểm tra:** kết quả trùng khớp `numpy.linalg.inv` tới sai số máy; hội tụ bậc hai thể hiện qua residual bình phương ở các vòng cuối.

In [ ]:
import numpy as np


# ── Định dạng số khoa học kiểu 1.23×10⁻⁵ ─────────────────────────
def _sci(x, sig=4):
    if abs(x) < 1e-14:
        return "0"
    s = f"{x:.{sig}e}"
    m, e = s.split("e")
    exp = int(e)
    sup = str(abs(exp)).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    sign = "⁻" if exp < 0 else ""
    return f"{m}×10{sign}{sup}"


def _print_mat(M, label, w=8, p=5):
    print(f"\n{label}:")
    for row in M:
        print("  " + "  ".join(f"{v:{w}.{p}f}" for v in row))


# ── Phương pháp chọn: lặp tựa Newton (Newton–Schulz) ─────────────
def inv_newton(A, eps=1e-10, max_iter=100, log=True):
    A = np.array(A, dtype=float)
    n = len(A)
    I = np.eye(n)
    X = A.T / (np.linalg.norm(A, 1) * np.linalg.norm(A, np.inf))
    if log:
        print(f"{'k':>3}  {'‖I - A·X‖∞':>14}")
    for k in range(1, max_iter + 1):
        r = np.max(np.abs(I - A @ X))
        X = X @ (2 * I - A @ X)
        if log:
            print(f"{k:>3}  {_sci(r):>14}")
        if r < eps:
            return X, k
    return X, max_iter


# ====== Ma trận A = C + a·I, a = 200 (đề Câu 42c) ======
if __name__ == "__main__":
    a = 200
    C = np.array([
        [ 11,  22, -13,  24,  15, -26,  17,  28],
        [ 22, 233,  24,  35,  26,  37,  28, -39],
        [ 33, -24,  35, -26,  37,  28, -39,  20],
        [ 14,  45,  26,  47,  38,  49,  40, -41],
        [-55,  16,  57,  28,  59,  30, -51,  42],
        [ 46,  27, -48,  39,  40,  61,  42,  73],
        [ 27, -58,  29,  70, -21,  42,  23,  34],
        [ 38,  59,  60, -71,  82, -93,  24,  15],
    ], dtype=float)
    A = C + a * np.eye(8)                       # a chỉ cộng vào đường chéo
    n = len(A)

    # ── Phân tích để chọn phương pháp ──
    off_r = np.sum(np.abs(A - np.diag(np.diag(A))), axis=1)
    print("Phân tích ma trận:")
    print(f"  Đối xứng?              {np.allclose(A, A.T)}")
    print(f"  Chéo trội nghiêm ngặt? {np.all(np.abs(np.diag(A)) > off_r)}")
    print(f"  Số điều kiện κ₂(A)     {np.linalg.cond(A):.4f}")
    print(f"  → Chọn: lặp tựa Newton (an toàn, không cần chéo trội/đối xứng)\n")

    X, k = inv_newton(A, eps=1e-10)
    print(f"\nSố vòng lặp: {k}")
    _print_mat(X, "Ma trận nghịch đảo A⁻¹ (tựa Newton)")

    # ── Kiểm tra & đánh giá sai số ──
    err_I  = np.max(np.abs(A @ X - np.eye(n)))
    err_np = np.max(np.abs(X - np.linalg.inv(A)))
    print("\n=== Kiểm tra & đánh giá ===")
    print(f"  ‖A·X - I‖∞          = {_sci(err_I)}")
    print(f"  Sai số vs numpy.inv = {_sci(err_np)}"
          f"  →  {'ĐÚNG ✓' if err_np < 1e-8 else 'SAI ✗'}")

    # ── Nhận xét: bán kính phổ ma trận lặp Jacobi / Gauss-Seidel ──
    D = np.diag(np.diag(A)); L = np.tril(A, -1); U = np.triu(A, 1)
    rho_j  = max(abs(np.linalg.eigvals(-np.linalg.inv(D) @ (L + U))))
    rho_gs = max(abs(np.linalg.eigvals(-np.linalg.inv(D + L) @ U)))
    print(f"\n  ρ(T_Jacobi)      = {rho_j:.4f}")
    print(f"  ρ(T_GaussSeidel) = {rho_gs:.4f}")
    print(f"  → Cả hai < 1: dù A KHÔNG chéo trội, Jacobi/GS vẫn hội tụ")
    print(f"    (chéo trội chỉ là điều kiện ĐỦ, không cần thiết).")
